# Derive Automatic Labels

## Import libs

In [ ]:
import os
import geopandas as gpd
from rasterstats import zonal_stats
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import xarray as xr
from skimage.filters import threshold_otsu 
import xrspatial 
from scipy import ndimage
from pathlib import Path

## Define Paths

In [ ]:
notebook_dir = Path.cwd()
base_path = notebook_dir.parent.parent / "Data" 
base_path_str = str(base_path)
print(f"Base path automatically set to: {base_path_str}")

In [ ]:
# Load image
indices_dir = os.path.join(base_path, "8_Images_with_Indices")
image_path = os.path.join(indices_dir, "250821_7_channel.tif")
image=xr.open_dataset(image_path,engine="rasterio",band_as_variable=True)

In [ ]:
# Load .shp file
shp_dir = os.path.join(base_path, "7_OBIA_Segmentation")
gdf = gpd.read_file(os.path.join(shp_dir,"Vectorized_r3.shp"))

## OTSU Threshold Classification

### 1. Identify Trees using CHM

In [ ]:
# Get CHM
chm = image.band_4

# Plot CHM
plt.clf()
chm.plot(cmap = "viridis")
plt.title("CHM")
plt.show()

# Get Otsu Threshold
chm_values = chm.values.flatten()
nans = np.isnan(chm_values).sum()
print("NaN proportion", nans / chm_values.shape[0])
chm_values = chm_values[~np.isnan(chm_values)]
chm_threshold = threshold_otsu(chm_values)
print("Otsu Threshold: ", chm_threshold)

# Create Mask based on threshold
mask_chm=xrspatial.reclassify(chm, bins=[chm_threshold, float(chm.max().values)],new_values=[0,1])

# Plot idenfified trees
mask_chm = mask_chm == 1
plt.clf()
mask_chm.plot(cmap='gray')
plt.show()

In [ ]:
# 1. Get the binary mask as a pure NumPy array
mask_data = mask_chm.values

# 2. Fill holes inside the canopy raster
mask_chm_filled_data = ndimage.binary_fill_holes(mask_data)

# 3. QUANTITATIVE CHECK: Calculate pixel statistics
pixels_before = np.sum(mask_data)
pixels_after = np.sum(mask_chm_filled_data)
pixels_added = pixels_after - pixels_before

print("==================================================")
print("🕳️ CANOPY HOLE FILLING METRICS")
print("==================================================")
print(f"Canopy pixels BEFORE filling: {pixels_before:,}")
print(f"Canopy pixels AFTER filling:  {pixels_after:,}")
print(f"Total pixels added (gaps filled): {pixels_added:,}")
if pixels_before > 0:
    print(f"Relative area increase:       {((pixels_added / pixels_before) * 100):.2f}%")
print("==================================================\n")

# 4. VISUAL CHECK: Create a difference mask to see EXACTLY where pixels changed
# This is True only where a hole was filled (After is True, but Before was False)
mask_added_data = mask_chm_filled_data & (~mask_data)

# Convert back to xarray format to preserve spatial coordinates
mask_chm_filled = xr.DataArray(mask_chm_filled_data, coords=mask_chm.coords, dims=mask_chm.dims)
mask_chm_added = xr.DataArray(mask_added_data, coords=mask_chm.coords, dims=mask_chm.dims)

# 5. Plot Before, After, and the isolated changes side-by-side
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Original Mask
mask_chm.plot(ax=axes[0], cmap='gray', add_colorbar=False)
axes[0].set_title("Before (With gaps in canopy)")
axes[0].axis("off")

# Panel 2: Filled Mask
mask_chm_filled.plot(ax=axes[1], cmap='gray', add_colorbar=False)
axes[1].set_title("After (Filled Gaps)")
axes[1].axis("off")

# Panel 3: Added Pixels (The "Gains")
mask_chm_added.plot(ax=axes[2], cmap='hot', add_colorbar=False)
axes[2].set_title(f"Isolated Gaps Filled (+{pixels_added:,} px)")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## Save plots

In [ ]:
save_dir = os.path.join(base_path.parent, "docs", "images")
print(save_dir)

In [ ]:
# Create the figure object without axes or margins
# fig = plt.figure(figsize=(mask_chm_filled.shape[1] / 100, mask_chm_filled.shape[0] / 100), dpi=100)
fig = plt.figure(figsize=(20, 6), dpi=100)
ax = plt.Axes(fig, [0., 0., 1., 1.])
ax.set_axis_off()
fig.add_axes(ax)

# Plot the array
ax.imshow(mask_chm_filled.values, cmap='gray')

# Save without any whitespace or borders
save_path = os.path.join(save_dir, "pixel_trees_chm.png")
plt.savefig(save_path, bbox_inches='tight', pad_inches=0, dpi=300)
plt.close()

### 2. Identify healthy vine using ExG

### 2.1 First Identify healthy vegetation

In [ ]:
# Get ExG
exg = image.band_5

# Plot ExG
plt.clf()
exg.plot(cmap = "viridis")
plt.title("ExG")
plt.show()

# Get Otsu Threshold
exg_values = exg.values.flatten()
nans = np.isnan(exg_values).sum()
print("NaN proportion", nans / exg_values.shape[0])
exg_values = exg_values[~np.isnan(exg_values)]
exg_threshold = threshold_otsu(exg_values)
print("Otsu Threshold: ", exg_threshold)

# Create Mask based on threshold
mask_exg=xrspatial.reclassify(exg, bins=[exg_threshold, float(exg.max().values)],new_values=[0,1])

# Plot identified vegetation
mask_exg = mask_exg == 1
plt.clf()
mask_exg.plot(cmap='gray')
plt.show()

### 2.2 Substract identified trees from vegetation pixels to obtain Vine pixles

In [ ]:
# --- 1. Tree Mask ---
mask_trees_pixel = mask_chm_filled_data.astype(bool)

# --- 2. ExG MASK  ---
mask_exg_pixel = mask_exg.values.astype(bool)

# --- 3. SUBSTRACTION ---
# Logic: It's vine if it's green (mask_exg_pixel) AND NOT (~) a tree  (~mask_trees_pixel).
grass_chm_thresh = 0.1
mask_taller_than_grass = (chm.values > grass_chm_thresh)

mask_vines_pixel = mask_exg_pixel & ~mask_trees_pixel & mask_taller_than_grass

mask_vines_xr = xr.DataArray(mask_vines_pixel, coords=mask_exg.coords, dims=mask_exg.dims)

# --- PLOT ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(mask_trees_pixel, cmap='gray')
axes[0].set_title("1. Trees (CHM)")
axes[0].axis('off')

axes[1].imshow(mask_exg_pixel, cmap='gray')
axes[1].set_title("2. Vegetation (ExG)")
axes[1].axis('off')

axes[2].imshow(mask_vines_pixel, cmap='gray')
axes[2].set_title("3. Only Vine (Vegetation MINUS Trees)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

## Save plots

In [ ]:
# Create the figure object without axes or margins
fig = plt.figure(figsize=(20, 6), dpi=100)
ax = plt.Axes(fig, [0., 0., 1., 1.])
ax.set_axis_off()
fig.add_axes(ax)

# Plot the array
ax.imshow(mask_exg_pixel, cmap='gray')

# Save without any whitespace or borders
save_path = os.path.join(save_dir, "pixel_healthy_exg.png")
plt.savefig(save_path, bbox_inches='tight', pad_inches=0, dpi=300)
plt.close()

In [ ]:
fig = plt.figure(figsize=(20, 6), dpi=100)
ax = plt.Axes(fig, [0., 0., 1., 1.])
ax.set_axis_off()
fig.add_axes(ax)

# Plot the array
ax.imshow(mask_vines_pixel, cmap='gray')

# Save without any whitespace or borders
save_path = os.path.join(save_dir, "pixel_vine_isolated.png")
plt.savefig(save_path, bbox_inches='tight', pad_inches=0, dpi=300)
plt.close()

## Transfer from Pixel to Polygon Logic

In [ ]:
# Load image as rasterio
with rasterio.open(image_path) as src:
    transform = src.transform

# Convert bool masks to 0 and 1 masks
trees_int = mask_trees_pixel.astype(np.uint8)
vines_int = mask_vines_pixel.astype(np.uint8)

print("Calculate Polygon-Overlap for Trees...")
stats_trees = zonal_stats(gdf, trees_int, affine=transform, stats="mean")

print("Calculate Polygon-Overlap for Vines...")
stats_vines = zonal_stats(gdf, vines_int, affine=transform, stats="mean")

# Write as new columns to Geo Data Frame
gdf['tree_cov'] = [s['mean'] if s['mean'] is not None else 0 for s in stats_trees]
gdf['vine_cov'] = [s['mean'] if s['mean'] is not None else 0 for s in stats_vines]

print("Finished!")

In [ ]:
# Create new column
gdf['Final_Class'] = -1

# If a polygon is filled more than 80% with pixel of one class label it as Tree/Vine
mask_is_tree = gdf['tree_cov'] > 0.8
gdf.loc[mask_is_tree, 'Final_Class'] = 20

mask_is_vine = gdf['vine_cov'] > 0.8
gdf.loc[mask_is_vine, 'Final_Class'] = 10

# Save
out_dir = os.path.join(base_path, "9_Training_Data")
gdf.to_file(os.path.join(out_dir, "automated_training_labels.shp"))

print(f"Number of labeled Tree polygons: {mask_is_tree.sum()}")
print(f"number of labeled Vine polygons: {mask_is_vine.sum()}")

## Save Polygon plots

In [ ]:
def save_clean_plot(gdf, filter_condition, color, filename):
    fig = plt.figure(figsize=(10, 10), dpi=300)
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)
    
    gdf.plot(ax=ax, color='lightgray', edgecolor='none')
    
    if filter_condition is not None:
        gdf[filter_condition].plot(ax=ax, color=color)
    
    plt.savefig(os.path.join(save_dir, filename), bbox_inches='tight', pad_inches=0, dpi=300)
    plt.close()

# 1. Tree Pixels Plot
save_clean_plot(gdf, gdf['Final_Class'] == 20, 'forestgreen', "poly_trees.png")

# 2. Vine Pixels Plot
save_clean_plot(gdf, gdf['Final_Class'] == 10, 'darkgoldenrod', "poly_vines.png")

# 3. Combined Overview
fig = plt.figure(figsize=(10, 10), dpi=300)
ax = plt.Axes(fig, [0., 0., 1., 1.])
ax.set_axis_off()
fig.add_axes(ax)
gdf.plot(ax=ax, color='lightgray', edgecolor='none')
gdf[gdf['Final_Class'] == 20].plot(ax=ax, color='forestgreen')
gdf[gdf['Final_Class'] == 10].plot(ax=ax, color='darkgoldenrod')
plt.savefig(os.path.join(save_dir, "combined_overview.png"), bbox_inches='tight', pad_inches=0, dpi=300)
plt.close()